## 2 — Gráficos: Visão Geral do Campus Restinga

| Gráfico | Tipo | Descrição |
|--------|------|-----------|
| 1 | Linha | Evolução de TC, TE, TR, TPE, IEf e TEFAcad por ano |
| 2 | Linha | Evolução de MREG e MRET por ano |
| 3 | Barras empilhadas | Matrículas atendidas por ano e categoria |
| 4 | Barras empilhadas | Matrículas por curso e ano |
| 5 | Linha | Ingressantes e Concluintes por ano |



### 2.1. Importações e configurações

In [24]:
import pandas as pd
import numpy as np
import plotly.express as px

pd.set_option("display.max_columns", None)

# Paleta de cores por categoria de situação
CORES_CATEGORIA = {
    "Em curso":    "#4CAF50",
    "Concluintes": "#2196F3",
    "Evadidos":    "#F44336",
}

# Paleta de cores por situação de matrícula detalhada
CORES_SITUACAO = {
    "Ingressante":          "#A5D6A7",
    "Em fluxo":             "#4CAF50",
    "Retido":               "#FF9800",
    "Concluída no prazo":   "#2196F3",
    "Concluída com atraso": "#64B5F6",
    "Abandono":             "#F44336",
    "Desligada":            "#E91E63",
    "Transf. externa":      "#9E9E9E",
    "Transf. interna":      "#BDBDBD",
    "Integralizada":        "#00BCD4",
    "Reprovada":            "#795548",
}

# Paleta de cores por indicador percentual
CORES_INDICADORES = {
    "TC":      "#2196F3",
    "TE":      "#F44336",
    "TR":      "#FF9800",
    "IEf":     "#4CAF50",
    "TPE":     "#9C27B0",
    "TEFAcad": "#00BCD4",
}

### 2.2 Carga dos dados tratados

In [25]:
df_dados_tratados = pd.read_parquet("../dados_tratados/restinga_dados_tratados.parquet")

print("Shape:", df_dados_tratados.shape)
df_dados_tratados.head(3)

Shape: (10445, 19)


,Ano,Categoria da Situação,Cor / Raça,Código da Matricula,Código do Ciclo Matricula,Data de Fim Previsto do Ciclo,Data de Inicio do Ciclo,Data de Ocorrencia da Matricula,Eixo Tecnológico,Faixa Etária,Mês De Ocorrência da Situação,Nome de Curso,Renda Familiar,Sexo,Situação de Matrícula,Subeixo Tecnológico,Tipo de Curso,Turno,Concluinte_Previsto
0,2017,Em curso,Não declarada,44681986,1207788,2014-12-22,2012-03-05,2012-03-01,Informação e Comunicação,35 a 39 anos,2018-03-05,Análise e Desenvolvimento de Sistemas,Não declarada,F,Retido,Informática,Superior-Tecnologia,Matutino,False
1,2017,Em curso,Branca,66262145,2007734,2019-12-20,2016-02-22,2016-02-01,Controle e Processos Industriais,15 a 19 anos,2016-02-01,Técnico em Eletrônica,"0,5<RFP<=1",F,Em fluxo,Elétrica,Técnico-Integrado,Matutino,False
2,2017,Em curso,Branca,66262155,2007734,2019-12-20,2016-02-22,2016-02-01,Controle e Processos Industriais,15 a 19 anos,2016-02-01,Técnico em Eletrônica,"0,5<RFP<=1",F,Em fluxo,Elétrica,Técnico-Integrado,Matutino,False


In [26]:
df_dados_tratados.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10445 entries, 0 to 10444
Data columns (total 19 columns):
 #   Column                           Non-Null Count  Dtype         
---  ------                           --------------  -----         
 0   Ano                              10445 non-null  int64         
 1   Categoria da Situação            10445 non-null  object        
 2   Cor / Raça                       10445 non-null  object        
 3   Código da Matricula              10445 non-null  int64         
 4   Código do Ciclo Matricula        10445 non-null  int64         
 5   Data de Fim Previsto do Ciclo    10445 non-null  datetime64[ns]
 6   Data de Inicio do Ciclo          10445 non-null  datetime64[ns]
 7   Data de Ocorrencia da Matricula  10445 non-null  datetime64[ns]
 8   Eixo Tecnológico                 10445 non-null  object        
 9   Faixa Etária                     10445 non-null  object        
 10  Mês De Ocorrência da Situação    10445 non-null  datetime6

### 2.3. Cálculo dos indicadores

In [27]:
def calcular_indicadores(df, agrupamento):

    df_indicadores = (
        df.groupby(agrupamento)
        .agg(
            ingressantes    = ("Situação de Matrícula", lambda x: (x == "Ingressante").sum()),
            em_curso        = ("Categoria da Situação",  lambda x: (x == "Em curso").sum()),
            concluintes     = ("Categoria da Situação",  lambda x: (x == "Concluintes").sum()),
            evadidos        = ("Categoria da Situação",  lambda x: (x == "Evadidos").sum()),
            matr_atendidas  = ("Código da Matricula",    "count"),
            conc_no_prazo   = ("Situação de Matrícula",  lambda x: (x == "Concluída no prazo").sum()),
            conc_com_atraso = ("Situação de Matrícula",  lambda x: (x == "Concluída com atraso").sum()),
            abandono        = ("Situação de Matrícula",  lambda x: (x == "Abandono").sum()),
            desligados      = ("Situação de Matrícula",  lambda x: (x == "Desligada").sum()),
            transf_ext      = ("Situação de Matrícula",  lambda x: (x == "Transf. externa").sum()),
            transf_int      = ("Situação de Matrícula",  lambda x: (x == "Transf. interna").sum()),
            integralizadas  = ("Situação de Matrícula",  lambda x: (x == "Integralizada").sum()),
            conc_previstos  = ("Concluinte_Previsto",    "sum"),
            MREG            = ("Situação de Matrícula",
                               lambda x: ((x == "Em fluxo") | (x == "Ingressante")).sum()),
            MRET            = ("Situação de Matrícula",  lambda x: (x == "Retido").sum()),
        )
        .reset_index()
    )

    # Evita divisão por zero substituindo 0 por NaN antes de dividir
    ma = df_indicadores["matr_atendidas"].replace(0, np.nan)

    # TC — Taxa de Conclusão
    df_indicadores["TC"] = (
        (df_indicadores["conc_no_prazo"] + df_indicadores["conc_com_atraso"]) / ma * 100
    )

    # TE — Taxa de Evasão
    df_indicadores["TE"] = (
        (df_indicadores["abandono"] + df_indicadores["desligados"] + df_indicadores["transf_ext"])
        / ma * 100
    )

    # TR — Taxa de Retenção
    df_indicadores["TR"] = df_indicadores["MRET"] / ma * 100

    # IEf — Índice de Eficiência
    # Numerador: todos os concluintes (no prazo + com atraso)
    # Denominador: matrículas que tiveram algum desfecho (todas as saídas)
    matr_finalizadas = (
        df_indicadores["conc_no_prazo"] + df_indicadores["conc_com_atraso"]
        + df_indicadores["abandono"] + df_indicadores["desligados"]
        + df_indicadores["transf_int"] + df_indicadores["transf_ext"]
    ).replace(0, np.nan)
    df_indicadores["IEf"] = (
        (df_indicadores["conc_no_prazo"] + df_indicadores["conc_com_atraso"])
        / matr_finalizadas * 100
    )

    # t_matr_cont_reg — Taxa de Matrícula Continuada Regular - para o cálculo da TPE
    df_indicadores["t_matr_cont_reg"] = df_indicadores["MREG"] / ma * 100

    # TPE — Taxa de Permanência e Êxito
    df_indicadores["TPE"] = df_indicadores["TC"] + df_indicadores["t_matr_cont_reg"]

    # TEFAcad — Taxa de Efetividade Acadêmica
    df_indicadores["TEFAcad"] = (
        df_indicadores["conc_no_prazo"]
        / df_indicadores["conc_previstos"].replace(0, np.nan) * 100
    )

    return df_indicadores.fillna(0).round(2)


# Calcula por Ano e por Ano + Curso
ind_ano       = calcular_indicadores(df_dados_tratados, ["Ano"])
ind_ano_curso = calcular_indicadores(df_dados_tratados, ["Ano", "Nome de Curso"])

print(ind_ano[["Ano", "TC", "TE", "TR", "TPE", "TEFAcad", "IEf"]])


    Ano    TC     TE     TR    TPE  TEFAcad    IEf
0  2017  8.38  15.78   5.18  78.55    45.71  34.34
1  2018  7.80   9.84   7.70  82.36    43.23  43.96
2  2019  6.07  11.05   5.68  83.19    26.87  35.45
3  2020  0.16   7.52  13.57  78.66     0.00   2.06
4  2021  2.50   5.70  23.89  70.41     0.00  30.43
5  2022  7.48   9.29  34.60  56.11    15.77  44.62
6  2023  9.77   5.27  31.94  62.79    47.78  64.94
7  2024  5.91   2.32  35.15  62.42    26.42  71.81


### 2.4. Gráficos

In [28]:
# Gráfico 1: Evolução de todos os indicadores percentuais por ano
# melt() transforma o DataFrame de formato largo (uma coluna por indicador) para formato longo (uma linha por indicador+ano)

ind_longo = ind_ano.melt(
    id_vars="Ano",
    value_vars=["TC", "TE", "TR", "TPE", "IEf", "TEFAcad"],
    var_name="Indicador",
    value_name="Valor (%)",
)

fig_g1 = px.line(
    ind_longo,
    x="Ano",
    y="Valor (%)",
    color="Indicador",
    markers=True,
    color_discrete_map=CORES_INDICADORES,
    title="1 — Evolução dos Indicadores Acadêmicos do Campus Restinga",
    labels={"Indicador": "", "Valor (%)": "(%)"},
)
fig_g1.update_xaxes(tickmode="linear", dtick=1)
fig_g1.update_layout(hovermode="x unified", legend=dict(orientation="h", y=-0.2))
fig_g1.show()

In [29]:
# Gráfico 2: Evolução de MREG e MRET por ano
# MREG = matrículas ativas regulares (Em fluxo + Ingressantes)
# MRET = matrículas ativas retidas (além do prazo previsto)
# São contagens absolutas, por isso coloquei em gráfico separado dos indicadores

matr_ativas_longo = ind_ano.melt(
    id_vars="Ano",
    value_vars=["MREG", "MRET"],
    var_name="Tipo",
    value_name="Matrículas",
)

fig_g2 = px.line(
    matr_ativas_longo,
    x="Ano",
    y="Matrículas",
    color="Tipo",
    markers=True,
    color_discrete_map={"MREG": "#4CAF50", "MRET": "#FF9800"},
    title="2 — Matrículas Ativas Regulares (MREG) e Retidas (MRET) por Ano",
    labels={"Tipo": ""},
)
fig_g2.update_xaxes(tickmode="linear", dtick=1)
fig_g2.update_layout(hovermode="x unified", legend=dict(orientation="h", y=-0.2))
fig_g2.show()

In [30]:
# Gráfico 3: Matrículas atendidas por ano e categoria

mat_cat = (
    df_dados_tratados.groupby(["Ano", "Categoria da Situação"])["Código da Matricula"]
    .count()
    .reset_index(name="Qtd")
)

fig_g3 = px.bar(
    mat_cat,
    x="Ano",
    y="Qtd",
    color="Categoria da Situação",
    barmode="stack",
    color_discrete_map=CORES_CATEGORIA,
    title="3 — Matrículas Atendidas por Ano e Categoria",
    labels={"Qtd": "Matrículas", "Categoria da Situação": "Categoria"},
    text_auto=True,
)
fig_g3.update_xaxes(tickmode="linear", dtick=1)
fig_g3.show()

In [31]:
# Gráfico 4: Matrículas atendidas por curso e ano

mat_curso_ano = (
    df_dados_tratados.groupby(["Ano", "Nome de Curso"])["Código da Matricula"]
    .count()
    .reset_index(name="Qtd")
)

fig_g4 = px.bar(
    mat_curso_ano,
    x="Ano",
    y="Qtd",
    color="Nome de Curso",
    barmode="stack",
    title="4 — Matrículas Atendidas por Curso e Ano",
    labels={"Qtd": "Matrículas", "Nome de Curso": "Curso"},
)
fig_g4.update_xaxes(tickmode="linear", dtick=1)
fig_g4.show()

In [32]:
# Gráfico 5: Ingressantes e Concluintes por ano

fluxo_longo = ind_ano.melt(
    id_vars="Ano",
    value_vars=["ingressantes", "concluintes"],
    var_name="Tipo",
    value_name="Quantidade",
)

# Renomeia os rótulos para ficarem mais legíveis na legenda
fluxo_longo["Tipo"] = fluxo_longo["Tipo"].map({
    "ingressantes": "Ingressantes",
    "concluintes":  "Concluintes",
})

fig_g5 = px.line(
    fluxo_longo,
    x="Ano",
    y="Quantidade",
    color="Tipo",
    markers=True,
    color_discrete_map={
        "Ingressantes": "#7E57C2",
        "Concluintes":  "#2196F3",
    },
    title="5 — Ingressantes e Concluintes por Ano",
    labels={"Tipo": ""},
)
fig_g5.update_xaxes(tickmode="linear", dtick=1)
fig_g5.update_layout(hovermode="x unified", legend=dict(orientation="h", y=-0.2))
fig_g5.show()